# AI basics lab 1: how an LLM works

Run the cells top to bottom with Shift+Enter.

In [4]:
import os

import matplotlib.pyplot as plt

from ai_basics import experiments as ex
from ai_basics.llm import FakeClient, OllamaClient

# Which model to talk to. The defaults use your local Ollama.
# Set AI_BASICS_PROVIDER=fake to run everything without a model (this is what CI does).
provider = os.environ.get("AI_BASICS_PROVIDER", "ollama")
model = os.environ.get("AI_BASICS_MODEL", "qwen2.5:3b")

if provider == "fake":
    client = FakeClient()
else:
    client = OllamaClient(model=model)

print(f"provider: {provider} | model: {client.model}")

provider: ollama | model: qwen2.5:3b


## Part A. Tokens: what the model actually reads

In [5]:
# The five texts from the guide's Predict box.
TEXTS = [
    "The cat sat on the mat.",
    "unbelievably",
    "def binary_search(arr, target):",
    "El gato se sentó en la alfombra.",
    "1234567",
]

# For each text: how many tokens the model read, how many characters it has,
# and how many characters fit in one token on average.
print(f"{'tokens':>6}  {'chars':>5}  {'chars/token':>11}  text")
for text in TEXTS:
    n_tokens = ex.count_tokens(client, text)
    n_chars = len(text)
    print(f"{n_tokens:>6}  {n_chars:>5}  {n_chars / n_tokens:>11.2f}  {text!r}")

tokens  chars  chars/token  text
     7     23         3.29  'The cat sat on the mat.'
     3     12         4.00  'unbelievably'
     7     31         4.43  'def binary_search(arr, target):'
    12     32         2.67  'El gato se sentó en la alfombra.'
     7      7         1.00  '1234567'


In [6]:
# Ask the model to repeat each text; every piece of its reply is one token.
# "|" marks where the text was cut, and · is a space.
for text in TEXTS:
    print(ex.show_tokens(ex.echo_tokens(client, text)))

The|·cat|·sat|·on|·the|·mat|.
un|belie|vably
def|·binary|_search|(arr|,|·target|):
El|·g|ato|·se|·sent|ó|·en|·la|·alf|omb|ra|.
1|2|3|4|5|6|7


## Part B. Next-token prediction

In [7]:
# The model's top 5 guesses for the next token, with their probabilities.
# repr() shows spaces inside a token, so ' Paris' and 'Paris' look different.
PROMPTS = [
    "The capital of France is",
    "Binary search runs in O(",
    "My favorite programming language is",
]

for prompt in PROMPTS:
    print(prompt)
    for token, p in ex.next_token_table(client, prompt, k=5):
        print(f"  {p:.3f}  {token!r}")
    print()

The capital of France is


AttributeError: module 'ai_basics.experiments' has no attribute 'next_token_table'

In [8]:
# A bar chart of the top 5 guesses after "Binary search runs in O(".
table = ex.next_token_table(client, "Binary search runs in O(", k=5)
labels = [repr(token) for token, _ in table]
probs = [p for _, p in table]

plt.bar(labels, probs)
plt.ylim(0, 1)  # probabilities go from 0 to 1
plt.xlabel("next token")
plt.ylabel("probability")
plt.title("After 'Binary search runs in O(' the model guesses...")
plt.show()

AttributeError: module 'ai_basics.experiments' has no attribute 'next_token_table'

In [9]:
# Your own softmax from sampling.py, on the paper example from the guide.
# T=0 is greedy, T=0.5 is sharper, T=1 is the original, T=2 is flatter.
from ai_basics.sampling import softmax_with_temperature

for t in [0, 0.5, 1.0, 2.0]:
    probs = softmax_with_temperature([2.0, 1.0, 0.0], t)
    print(f"T={t}: {[round(p, 3) for p in probs]}")

T=0: [1.0, 0.0, 0.0]
T=0.5: [0.867, 0.117, 0.016]
T=1.0: [0.665, 0.245, 0.09]
T=2.0: [0.506, 0.307, 0.186]


## Part C. Temperature in practice

In [10]:
# Ask the same question 5 times at temperature 0 and 5 times at temperature 1.
# set() keeps only the different answers, so len(set(...)) counts them.
QUESTION = "Suggest one name for a pet turtle. Reply with the name only."

for t in [0.0, 1.0]:
    answers = ex.repeat_answers(client, QUESTION, 5, t)
    print(f"T={t}: {len(set(answers))} different answers")
    for answer in answers:
        print(f"  {answer}")
    print()

AttributeError: module 'ai_basics.experiments' has no attribute 'repeat_answers'